# CS781 Phase 3 - Rhetorical Role Prediction on top of the Phase-1 models

| switch | options |
|---|---|
| `MODEL_TYPE` | `"gpt"`, `"modern"`, `"rnn"`, `"t5"` (your Phase-1 models; for `t5` only the encoder is used) |
| `FT_METHOD` | `"full"`, `"lora"`, `"rslora"`, `"lorma"`, `"ia3"`, `"partial"` (top layers only), `"head_only"` (linear probe) |
| `USE_KD` | `True`: train a smaller student (first `STUDENT_LAYERS` layers of `MODEL_TYPE`) from a saved teacher |
| `CONTEXT_MODE` | `"sentence"`: every sentence alone; `"document"`: windows of neighbouring sentences |
| `SENT_CONTEXT` | `"none"` or `"bigru"`: a bidirectional GRU over the sentence vectors of a window |

Pipeline: document -> sentences -> BPE tokens (Phase-1 tokenizer, `</s>` after every sentence) -> windows of whole sentences -> Phase-1 model -> one vector per sentence (mean of its token states) -> (optional) bidirectional GRU over the sentences -> linear layer -> one label per sentence -> `prediction.json`.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import gc

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
DOMAINS = ["CL", "IT"]
TRAIN_FILES = {d: f"Phase_3_RR/Train/{d}_train.json" for d in DOMAINS}
DEV_FILES = {d: f"Phase_3_RR/Train/{d}_dev.json" for d in DOMAINS}
TEST_FILES = {d: f"Phase_3_RR/Test/{d}_test.json" for d in DOMAINS}
LABEL_VOCAB_FILE = "Phase_3_RR/Train/label_vocab.json"

# ---------------- the experiment ----------------
EXPERIMENT_NAME = "modern_lorma_doc"    # <-- CHANGED for this run
MODEL_TYPE = "modern"    # <-- CHANGED for this run
FT_METHOD = "lorma"    # <-- CHANGED for this run
SMOKE_TEST = False
TRAIN_DOMAINS = ["CL", "IT"] # train on both domains together; ["CL"] or ["IT"] = one domain only
USE_PHASE1_WEIGHTS = True    # False = random initialisation (ablation only: is Phase-1 pre-training useful?)

# context
CONTEXT_MODE = "document"    # "sentence": every sentence alone | "document": windows of neighbouring sentences
WINDOW_TOKENS = 512          # "document": at most this many tokens per window
MAX_SENT_TOKENS = 128        # long sentences are cut to their first MAX_SENT_TOKENS tokens
MAX_SENTS = 32               # "document": at most this many sentences per window
SENT_POOL = "mean"           # sentence vector: "mean" of its token states, or "last" (state of its </s> token)
SENT_CONTEXT = "bigru"       # "none" | "bigru": bidirectional GRU over the sentence vectors of a window (Module 2.1)
SENT_GRU_HIDDEN = 256
CLASS_WEIGHTING = "sqrt_inverse"   # "none" | "sqrt_inverse": loss weight of a label = sqrt(N / count)
CLASS_WEIGHT_CAP = 2.0       # no label gets more than 2x the average weight (PrecedentOverruled has only 3 training sentences)

# knowledge distillation (Section 9.2.2): student = first STUDENT_LAYERS layers of MODEL_TYPE
USE_KD = False
STUDENT_LAYERS = 2
TEACHER_MODEL_TYPE = "modern"
TEACHER_FT_METHOD = "full"
TEACHER_RR_PATH = "/kaggle/input/<teacher-dataset>/rr_model.pt"
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5               # loss = KD_ALPHA * CE(gold) + (1 - KD_ALPHA) * T^2 * KL(teacher || student)
# (the teacher must have been trained with the same CONTEXT_MODE, WINDOW_TOKENS, MAX_SENTS, SENT_POOL, SENT_CONTEXT)

# LoRA / rsLoRA / LoRMA
ADAPTER_RANK = 8
ADAPTER_ALPHA = 16
PARTIAL_TOP_LAYERS = 2       # "partial": only the top layers (+ final norm + head) are trained

# Phase-1 models: one Kaggle dataset with model_<type>.safetensors and bpe_legal_<type>.json
PHASE1_FOLDER = "/kaggle/input/datasets/medinim/phase1-models"   # searched automatically below if this path is wrong
if not os.path.exists(os.path.join(PHASE1_FOLDER, "model_modern.safetensors")):
  import glob
  hits = glob.glob("/kaggle/input/**/model_modern.safetensors", recursive=True)
  if hits:
    PHASE1_FOLDER = os.path.dirname(hits[0])
print("Phase-1 models from:", PHASE1_FOLDER)
PHASE1_MODEL_FILES = {k: os.path.join(PHASE1_FOLDER, f"model_{k}.safetensors") for k in ["gpt", "modern", "rnn", "t5"]}
PHASE1_BPE_FILES = {k: os.path.join(PHASE1_FOLDER, f"bpe_legal_{k}.json") for k in ["gpt", "modern", "rnn", "t5"]}

# Phase-1 architecture sizes: must be EXACTLY what Phase 1 trained with
MODEL_CONTEXT_LENGTH = 2048
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 640, 10, 10, 5
D_FF = 4 * N_EMBD
RNN_CELL, RNN_HIDDEN, RNN_LAYERS = "lstm", 1152, 4
T5_D_MODEL, T5_ENC_LAYERS, T5_DEC_LAYERS, T5_HEADS, T5_D_FF, T5_N_SENTINELS = 512, 8, 8, 8, 2048, 100
DROPOUT = 0.1
LOAD_RR_PATH = None          # path to a saved rr_model.pt -> skip training, only predict
REQUIRE_GPU = True           # stop at once if the notebook was started without a GPU

# training (Trainer, like Phase 1 and 2)
if FT_METHOD in ("full", "partial") or USE_KD:
  LEARNING_RATE = 1e-4
elif FT_METHOD == "ia3":
  LEARNING_RATE = 3e-3       # IA3 trains only scaling vectors; its paper uses a larger learning rate
else:
  LEARNING_RATE = 1e-3       # adapters and a fresh head can take a bigger learning rate
EPOCHS = 10
BATCH_SIZE = 8
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if CONTEXT_MODE == "sentence":
  WINDOW_TOKENS = MAX_SENT_TOKENS + 1   # one sentence + its </s>
  MAX_SENTS = 1
  BATCH_SIZE = 32
assert WINDOW_TOKENS >= MAX_SENT_TOKENS + 1

if SMOKE_TEST:
  EPOCHS = 1
if REQUIRE_GPU and not torch.cuda.is_available():
  raise RuntimeError("No GPU! In Kaggle: Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")
if USE_KD and not os.path.exists(TEACHER_RR_PATH):
  import glob
  hits = glob.glob("/kaggle/input/**/rr_model.pt", recursive=True)
  if hits:
    TEACHER_RR_PATH = hits[0]
  print("KD teacher:", TEACHER_RR_PATH)

OUTPUT_DIR = "/kaggle/working/cs781_phase3"
EXP_DIR = os.path.join(OUTPUT_DIR, EXPERIMENT_NAME)
os.makedirs(EXP_DIR, exist_ok=True)
PREDICTION_PATH = os.path.join(EXP_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, f"submission_{EXPERIMENT_NAME}.zip")
RESULTS_PATH = os.path.join(OUTPUT_DIR, "results.jsonl")
CAUSAL = MODEL_TYPE in ("gpt", "modern", "rnn")   # left-to-right models only see the past
print("Experiment:", EXPERIMENT_NAME, "|", MODEL_TYPE, FT_METHOD, CONTEXT_MODE, SENT_CONTEXT, "| KD" if USE_KD else "")

## 4. Tokenizer (the Phase-1 BPE, unchanged)

In [ ]:
import json
import heapq
from collections import Counter, defaultdict

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
WORD_END = "</w>"

class ByteLevelBPE:
  def __init__(self, vocab_size):
      self.vocab_size = vocab_size
      self.merges = []
      self.merge_ranks = {}
      self.itos = []
      self.stoi = {}
      self._cache = {}
      self.pad_id = None
      self.bos_id = None
      self.eos_id = None

  def _word_to_symbols(self, word):
      byte_vals = word.encode("utf-8")
      symbols = [chr(b) for b in byte_vals]
      symbols.append(WORD_END)
      return symbols

  def train(self, sentences):
      word_freq = Counter()
      for sent in sentences:
          for word in sent.split():
              word_freq[word] += 1

      words = list(word_freq)
      freqs = [word_freq[w] for w in words]
      word_symbols = [self._word_to_symbols(w) for w in words]

      vocab_list = list(SPECIALS)
      vocab_seen = set(vocab_list)

      initial_symbols = {chr(b) for b in range(256)} | {WORD_END}

      for s in sorted(initial_symbols):
          if s not in vocab_seen:
              vocab_list.append(s)
              vocab_seen.add(s)

      # this loop recounts every pair every merge, which is far too slow
      # for the legal corpus. Replace it with incremental pair counts + pair_to_words.
      pair_counts = Counter()
      pair_to_words = defaultdict(set)
      for wi, symbols in enumerate(word_symbols):
          f = freqs[wi]
          for i in range(len(symbols) - 1):
              p = (symbols[i], symbols[i + 1])
              pair_counts[p] += f
              pair_to_words[p].add(wi)

      heap = [(-c, p) for p, c in pair_counts.items()]
      heapq.heapify(heap)

      while len(vocab_list) < self.vocab_size:
          best_pair, best_count = None, 0
          while heap:
              negc, p = heapq.heappop(heap)
              if pair_counts.get(p, 0) == -negc and -negc > 0:
                  best_pair, best_count = p, -negc
                  break
          if best_pair is None or best_count <= 1:
              break

          a, b = best_pair
          merged = a + b
          changed = set()

          for wi in list(pair_to_words[best_pair]):
              symbols = word_symbols[wi]
              f = freqs[wi]

              old_pairs = Counter(zip(symbols, symbols[1:]))   # positions, so overlaps count
              new_symbols = []
              i = 0
              while i < len(symbols):
                  if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                      new_symbols.append(merged)
                      i += 2
                  else:
                      new_symbols.append(symbols[i])
                      i += 1
              word_symbols[wi] = new_symbols
              new_pairs = Counter(zip(new_symbols, new_symbols[1:]))

              for p, c in old_pairs.items():
                  pair_counts[p] -= c * f
                  changed.add(p)
                  if p not in new_pairs:
                      pair_to_words[p].discard(wi)
              for p, c in new_pairs.items():
                  pair_counts[p] += c * f
                  changed.add(p)
                  pair_to_words[p].add(wi)

          for p in changed:
              c = pair_counts[p]
              if c > 0:
                  heapq.heappush(heap, (-c, p))
              else:
                  del pair_counts[p]
                  pair_to_words.pop(p, None)

          self.merges.append([a, b])
          self.merge_ranks[(a, b)] = len(self.merges) - 1

          if merged not in vocab_seen:
              vocab_list.append(merged)
              vocab_seen.add(merged)

      self.itos = vocab_list
      self.stoi = {tok: i for i, tok in enumerate(self.itos)}
      self.pad_id = self.stoi["<pad>"]
      self.bos_id = self.stoi["<s>"]
      self.eos_id = self.stoi["</s>"]
      return self

  def encode_word(self, word):
      if word in self._cache:
        return self._cache[word]
      symbols = self._word_to_symbols(word)
      while len(symbols) > 1:
          pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
          ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
          if not ranked:
              break
          _, i = min(ranked)
          symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

      self._cache[word] = symbols
      return symbols

  def encode_ids(self, sentence):
      unk = self.stoi["<unk>"]
      return [self.stoi.get(t, unk) for t in self.encode(sentence)]

  def decode(self, ids):
      text = "".join(self.itos[i] for i in ids if self.itos[i] not in SPECIALS)
      words = text.split(WORD_END)
      out = [bytes(ord(c) for c in w).decode("utf-8", errors="replace") for w in words if w]
      return " ".join(out)

  def encode(self, sentence):
      tokens = []
      for word in sentence.split():
          tokens.extend(self.encode_word(word))
      return tokens

  def save(self, path):
      with open(path, "w", encoding="utf-8") as f:
          json.dump({"merges": self.merges, "itos": self.itos,
                      "vocab_size": self.vocab_size}, f, ensure_ascii=False)

  @staticmethod
  def load_bpe(path):
      with open(path, "r", encoding="utf-8") as f:
          data = json.load(f)
      bpe = ByteLevelBPE(vocab_size=data["vocab_size"])
      bpe.merges = data["merges"]
      bpe.itos = data["itos"]
      bpe.stoi = {tok: i for i, tok in enumerate(bpe.itos)}
      bpe.merge_ranks = {(a, b): i for i, (a, b) in enumerate(bpe.merges)}
      bpe.pad_id = bpe.stoi["<pad>"]
      bpe.bos_id = bpe.stoi["<s>"]
      bpe.eos_id = bpe.stoi["</s>"]
      return bpe

In [ ]:
import hashlib
for k, f in PHASE1_BPE_FILES.items():   # all four must be the same tokenizer (needed for KD)
  print(k, hashlib.md5(open(f, "rb").read()).hexdigest())
bpe = ByteLevelBPE.load_bpe(PHASE1_BPE_FILES[MODEL_TYPE])
UNK_ID = bpe.stoi["<unk>"]
print("vocab size:", len(bpe.itos))

## 5. Download the Phase-3 data

Every file is read into a list of documents `{"id", "domain", "sents", "labels"}`. The brief shows a list of `{"id", "text": [...], "labels": [...]}`; a dictionary `{id: {...}}` is also accepted.

In [ ]:
from huggingface_hub import hf_hub_download
from collections import Counter

def load_json_file(fname):
  p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
  with open(p, "r", encoding="utf-8") as f:
    return json.load(f)

def read_docs(data, domain, with_labels):
  if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
    data = data["data"]
  if isinstance(data, dict):                    # {id: {...}} or {id: [sentences]}
    items = []
    for k, v in data.items():
      if isinstance(v, list):
        v = {"text": v}
      v = dict(v)
      v.setdefault("id", k)
      items.append(v)
    data = items
  docs = []
  for i, d in enumerate(data):
    doc_id = str(d.get("id", d.get("doc_id", i)))
    sents = d.get("text", d.get("sentences"))
    if isinstance(sents, str):
      raise ValueError(f"{domain} document {doc_id}: expected a list of sentences in 'text'")
    labels = d.get("labels") if with_labels else None
    if with_labels and (labels is None or len(labels) != len(sents)):
      raise ValueError(f"{domain} document {doc_id}: number of labels != number of sentences")
    docs.append({"id": doc_id, "domain": domain, "sents": [str(s) for s in sents], "labels": labels})
  return docs

def read_label_vocab(v):
  if isinstance(v, dict) and "labels" in v:
    v = v["labels"]
  if isinstance(v, dict):                       # {label: id} or {id: label}
    if all(isinstance(x, int) for x in v.values()):
      return [l for l, _ in sorted(v.items(), key=lambda kv: kv[1])]
    return [v[k] for k in sorted(v, key=lambda k: int(k))]
  return list(v)

LABELS = read_label_vocab(load_json_file(LABEL_VOCAB_FILE))
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
train_docs, dev_docs, test_docs = [], [], []
for d in DOMAINS:
  dev_docs += read_docs(load_json_file(DEV_FILES[d]), d, True)
  test_docs += read_docs(load_json_file(TEST_FILES[d]), d, False)
  if d in TRAIN_DOMAINS:
    train_docs += read_docs(load_json_file(TRAIN_FILES[d]), d, True)
if SMOKE_TEST:
  train_docs = train_docs[:6]

print("labels:", LABELS)
for name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
  for d in DOMAINS:
    ds = [x for x in docs if x["domain"] == d]
    print(f"{name:5s} {d}: {len(ds)} documents, {sum(len(x['sents']) for x in ds)} sentences")
for name, docs in [("train", train_docs), ("dev", dev_docs)]:
  for x in docs:
    for l in x["labels"]:
      if l not in LABEL2ID:
        raise ValueError(f"label {l!r} is not in label_vocab.json")
  for d in DOMAINS:
    c = Counter(l for x in docs if x["domain"] == d for l in x["labels"])
    print(f"\n{name} {d} label counts (class imbalance):", dict(c.most_common()))

## 6. Sentence tokens and windows of sentences

* Every sentence is tokenized with the Phase-1 BPE, cut to `MAX_SENT_TOKENS` tokens, and followed by `</s>`, so the model can see where a sentence ends.
* **`CONTEXT_MODE = "sentence"`:** each window holds one sentence (no context).
* **`CONTEXT_MODE = "document"`:** each window holds as many *whole* neighbouring sentences as fit in `WINDOW_TOKENS` (at most `MAX_SENTS`). The next window starts half-way through the previous one, so every sentence is seen with context on both sides. At prediction time each sentence takes its label from the window where it is most central (or, for a left-to-right model without the GRU, where it has the most sentences on its left).
* `sent_index[t]` says which sentence of the window token `t` belongs to (-1 = padding). Labels are one per sentence (-100 = no sentence).

In [ ]:
from datasets import Dataset

def tokenize_sentence(s):
  return bpe.encode_ids(s)[:MAX_SENT_TOKENS] + [bpe.eos_id]

def make_windows(sent_tok):
  # list of (first sentence, last sentence + 1) for one document
  n = len(sent_tok)
  windows = []
  start = 0
  while start < n:
    end, total = start, 0
    while end < n and end - start < MAX_SENTS and total + len(sent_tok[end]) <= WINDOW_TOKENS:
      total += len(sent_tok[end])
      end += 1
    windows.append((start, end))
    if end >= n:
      break
    start = start + max(1, (end - start) // 2)    # half overlap
  return windows

def window_example(sent_tok, labels, s, e):
  ids, sent_index = [], []
  for j in range(s, e):
    ids += sent_tok[j]
    sent_index += [j - s] * len(sent_tok[j])
  pad = WINDOW_TOKENS - len(ids)
  ids += [bpe.pad_id] * pad                    # right padding
  sent_index += [-1] * pad
  if labels is None:
    y = [0] * (e - s)
  else:
    y = [LABEL2ID[labels[j]] for j in range(s, e)]
  y += [-100] * (MAX_SENTS - (e - s))
  return ids, sent_index, y

doc_tok = {}
for x in train_docs + dev_docs + test_docs:
  doc_tok[(x["domain"], x["id"])] = [tokenize_sentence(s) for s in x["sents"]]

def make_examples(docs):
  xs, si, ys = [], [], []
  for x in docs:
    st = doc_tok[(x["domain"], x["id"])]
    for s, e in make_windows(st):
      a, b, c = window_example(st, x["labels"], s, e)
      xs.append(a); si.append(b); ys.append(c)
  return xs, si, ys

lens = sorted(len(t) for x in train_docs for t in doc_tok[(x["domain"], x["id"])])
n_cut = sum(1 for l in lens if l == MAX_SENT_TOKENS + 1)
print(f"tokens per sentence: median {lens[len(lens) // 2]}, 95% {lens[int(0.95 * len(lens))]}, max {lens[-1]} | cut sentences: {n_cut} ({100 * n_cut / len(lens):.1f}%)")

dev_sel = [x for x in dev_docs if x["domain"] in TRAIN_DOMAINS]   # checkpoint selection uses the trained domains
tr_x, tr_s, tr_y = make_examples(train_docs)
va_x, va_s, va_y = make_examples(dev_sel)
if SMOKE_TEST:
  tr_x, tr_s, tr_y = tr_x[:64], tr_s[:64], tr_y[:64]
train_dataset = Dataset.from_dict({"input_ids": tr_x, "sent_index": tr_s, "labels": tr_y})
val_dataset = Dataset.from_dict({"input_ids": va_x, "sent_index": va_s, "labels": va_y})
print(train_dataset)
print(val_dataset)
print("sentences per window (train):", round(sum(sum(1 for v in y if v != -100) for y in tr_y) / len(tr_y), 1))

def data_collator(features):
  return {"input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
          "sent_index": torch.tensor([f["sent_index"] for f in features], dtype=torch.long),
          "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long)}

## 7. Phase-1 model code (copied unchanged from the Phase-1 notebook)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Module 2.3:  RMS(x) = sqrt( (1/E) * sum_i x_i^2 + eps ),   RMSNorm(x)_i = gamma_i * x_i / RMS(x)"""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # learnable, no bias (beta)

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)


# ---- RoPE (Module 2.3 / 3.1, Assignment 3 "rope.py") ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    # theta_i = base^(-2(i-1)/d) for every pair of dimensions; angle = position m * theta_i
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([freqs, freqs], dim=-1)                                # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    # split the last dim in half and return (-x2, x1)
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos, sin: (B, 1, T, d_head)
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention (Module 2.3, Assignment 3).
    n_kv_heads == n_heads -> normal MHA,  n_kv_heads < n_heads -> grouped-query attention (GQA).
    past_kv = (k, v) of earlier tokens -> KV cache for generation (Module 2.3).
    attention_mask (B, S): 1 = real token, 0 = padding (Module 2.3: masks prevent attention over padding)."""
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.1, use_rope=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            cos = self.rope_cos[position_ids].unsqueeze(1)   # (B, 1, T, d_head)
            sin = self.rope_sin[position_ids].unsqueeze(1)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past_kv is not None:                  # KV cache: reuse keys/values of earlier tokens
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v)

        if self.n_rep > 1:                       # GQA: each k/v head is shared by n_rep query heads
            k = torch.repeat_interleave(k, self.n_rep, dim=1)
            v = torch.repeat_interleave(v, self.n_rep, dim=1)

        S = k.size(2)                             # number of keys = cached + new
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale        # (B, H, T, S)
        allowed = self.causal_mask[S - T:S, :S].unsqueeze(0).unsqueeze(0)  # (1, 1, T, S)
        if attention_mask is not None:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            # a left-padding position has no real token before it, so let every
            # position always see itself (otherwise its whole row would be masked)
            itself = torch.eye(S, dtype=torch.bool, device=x.device)[S - T:S, :S]
            allowed = allowed | itself.unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~allowed, float("-inf"))
        att = F.softmax(scores, dim=-1)
        att = self.attn_dropout(att)
        out = att @ v                                                      # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out, new_kv


class FeedForward(nn.Module):
    """Assignment 3: two linear layers with a GELU in between, plus dropout."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class SwiGLUFeedForward(nn.Module):
    """Module 3.1 names SwiGLU as the modern feed-forward activation:
    FFN(x) = W2( SiLU(W1 x) * (W3 x) ),  SiLU(a) = a * sigmoid(a)"""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w3 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a = self.w1(x)
        return self.dropout(self.w2(a * torch.sigmoid(a) * self.w3(x)))

In [ ]:
from dataclasses import dataclass


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 10
    n_heads: int = 10
    n_kv_heads: int = 10            # == n_heads for plain MHA
    d_ff: int = 2560
    max_seq_len: int = 2048
    dropout: float = 0.1
    pos_emb_type: str = "learned"   # "learned" (GPT-2 / BERT, Module 2.3) or "rope"
    norm_type: str = "layernorm"    # "layernorm" or "rmsnorm"
    ffn_type: str = "gelu"          # "gelu" or "swiglu"


class Block(nn.Module):
    """Pre-norm block (Module 2.3, used by GPT-2 and modern LLMs):  x + Attn(Norm(x)),  then  x + FFN(Norm(x))"""
    def __init__(self, cfg):
        super().__init__()
        if cfg.norm_type == "layernorm":
            self.ln1 = nn.LayerNorm(cfg.d_model)
            self.ln2 = nn.LayerNorm(cfg.d_model)
        else:
            self.ln1 = RMSNorm(cfg.d_model)
            self.ln2 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"))
        if cfg.ffn_type == "gelu":
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        else:
            self.ffn = SwiGLUFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        a, new_kv = self.attn(self.ln1(x), position_ids, attention_mask, past_kv)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, new_kv


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)    # learned absolute positions
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        if cfg.norm_type == "layernorm":
            self.ln_f = nn.LayerNorm(cfg.d_model)
        else:
            self.ln_f = RMSNorm(cfg.d_model)
        # Module 2.1 "Proper Weight Initialization": Xavier/Glorot for the embedding matrices.
        # PyTorch's default N(0, 1) embedding + weight tying gives huge first logits (start loss ~80)
        nn.init.xavier_uniform_(self.tok_emb.weight)
        if cfg.pos_emb_type == "learned":
            nn.init.xavier_uniform_(self.pos_emb.weight)

    def forward(self, input_ids, labels=None, attention_mask=None, position_ids=None,
                past_key_values=None, use_cache=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.tok_emb(input_ids)
        if self.cfg.pos_emb_type == "learned":
            x = x + self.pos_emb(position_ids)
        x = self.drop(x)

        new_past = []
        for i, block in enumerate(self.blocks):
            layer_past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, position_ids, attention_mask, layer_past)
            new_past.append(kv)
        x = self.ln_f(x)

        if use_cache:
            x = x[:, -1:, :]              # generation only needs the prediction for the last position
        logits = x @ self.tok_emb.weight.t()   # weight tying

        loss = None
        if labels is not None:
            # predict token t+1 from positions <= t (labels are shifted inside, like GPT-2)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_past}
        return {"loss": loss, "logits": logits}

In [ ]:
class LSTMCell(nn.Module):
    """One time step of an LSTM (Assignment 2): packed linear layers compute f, i, g, o at once."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2g = nn.Linear(input_size, 4 * hidden_size)
        self.h2g = nn.Linear(hidden_size, 4 * hidden_size)
        with torch.no_grad():
            self.x2g.bias[0:hidden_size] = 1.0   # forget gate bias close to 1 (Module 2.1)
            self.h2g.bias[0:hidden_size] = 0.0

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.x2g(x_t) + self.h2g(h_prev)
        f_pre, i_pre, g_pre, o_pre = gates.chunk(4, dim=-1)
        f_t = torch.sigmoid(f_pre)
        i_t = torch.sigmoid(i_pre)
        g_t = torch.tanh(g_pre)
        o_t = torch.sigmoid(o_pre)
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return (h_t, c_t)


class GRUCell(nn.Module):
    """One time step of a GRU with the Module 2.1 equations (layer names as in Assignment 2)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2z = nn.Linear(input_size, hidden_size)
        self.h2z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2r = nn.Linear(input_size, hidden_size)
        self.h2r = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2n = nn.Linear(input_size, hidden_size)
        self.h2n = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        z_t = torch.sigmoid(self.x2z(x_t) + self.h2z(h_prev))          # update gate
        r_t = torch.sigmoid(self.x2r(x_t) + self.h2r(h_prev))          # reset gate
        n_t = torch.tanh(self.x2n(x_t) + self.h2n(r_t * h_prev))       # candidate h~
        h_t = (1 - z_t) * h_prev + z_t * n_t
        return h_t


class RNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, cell_type="lstm", dropout=0.1):
        super().__init__()
        self.is_lstm = (cell_type == "lstm")
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        cell_cls = LSTMCell if self.is_lstm else GRUCell
        self.cells = nn.ModuleList([cell_cls(hidden_size, hidden_size) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.tok_emb.weight)     # Module 2.1: Xavier/Glorot initialisation

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None, use_cache=False):
        """past_key_values here = list of per-layer states (h, c) / h, carried during generation.
        attention_mask (B, T): at padding positions the state is NOT updated."""
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))
        new_states = []
        for l in range(self.num_layers):
            cell = self.cells[l]
            if past_key_values is not None:
                state = past_key_values[l]
            else:
                h = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
                state = (h, torch.zeros_like(h)) if self.is_lstm else h
            outputs = []
            for t in range(T):           # explicit unroll over time (BPTT through these steps)
                new_state = cell(x[:, t, :], state)
                if attention_mask is not None:
                    m = attention_mask[:, t].unsqueeze(1).to(x.dtype)     # 1 = real token, 0 = pad
                    if self.is_lstm:
                        new_state = (m * new_state[0] + (1 - m) * state[0], m * new_state[1] + (1 - m) * state[1])
                    else:
                        new_state = m * new_state + (1 - m) * state
                state = new_state
                outputs.append(state[0] if self.is_lstm else state)
            new_states.append(state)
            x = self.dropout(torch.stack(outputs, dim=1))    # input of the next layer

        if use_cache:
            x = x[:, -1:, :]
        logits = x @ self.tok_emb.weight.t()       # weight tying (Assignment 3)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_states}
        return {"loss": loss, "logits": logits}

In [ ]:
class T5RelativePositionBias(nn.Module):
    def __init__(self, n_heads, bidirectional, num_buckets=32, max_distance=128):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.rel_emb = nn.Embedding(num_buckets, n_heads)     # one scalar per (bucket, head)

    def bucket(self, relative_position):
        # relative_position = key_pos - query_pos
        n = -relative_position
        num_buckets = self.num_buckets
        ret = torch.zeros_like(n)
        if self.bidirectional:                 # encoder: half the buckets for keys on the right
            num_buckets = num_buckets // 2
            ret = ret + (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:                                  # decoder: only keys on the left exist
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2           # small offsets get their own bucket
        is_small = n < max_exact
        # bigger offsets: logarithmically wider buckets, everything beyond max_distance shares the last one
        large = max_exact + (torch.log(torch.clamp(n, min=1).float() / max_exact)
                             / math.log(self.max_distance / max_exact)
                             * (num_buckets - max_exact)).long()
        large = torch.clamp(large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        buckets = self.bucket(k_pos - q_pos)               # (q, k)
        bias = self.rel_emb(buckets)                       # (q, k, H)
        return bias.permute(2, 0, 1).unsqueeze(0)          # (1, H, q, k)


class T5Attention(nn.Module):
    """Self-attention (x_kv = x_q) or cross-attention (x_kv = encoder output)."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, position_bias=None, allowed=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.size(1)
        q = self.q_proj(x_q).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale
        if position_bias is not None:
            scores = scores + position_bias
        if allowed is not None:
            scores = scores.masked_fill(~allowed, float("-inf"))
        att = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class T5FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))   # dropout inside the FFN


class T5EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, position_bias, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, src_allowed))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class T5DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = T5Attention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, position_bias, causal_allowed, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, causal_allowed))   # masked self-attention
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, None, src_allowed))        # cross-attention
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x


class T5Model(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.shared_emb = nn.Embedding(vocab_size, d_model)    # encoder input, decoder input and output
        self.dropout = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList([T5EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.dec_layers = nn.ModuleList([T5DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)])
        self.enc_norm = RMSNorm(d_model)
        self.dec_norm = RMSNorm(d_model)
        self.enc_pos_bias = T5RelativePositionBias(n_heads, bidirectional=True)
        self.dec_pos_bias = T5RelativePositionBias(n_heads, bidirectional=False)
        nn.init.xavier_uniform_(self.shared_emb.weight)  # Module 2.1: Xavier/Glorot initialisation

    def encode(self, input_ids):
        S = input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]          # no attention to padding
        position_bias = self.enc_pos_bias(S, S, input_ids.device)
        x = self.dropout(self.shared_emb(input_ids))
        for layer in self.enc_layers:
            x = layer(x, position_bias, src_allowed)
        return self.dropout(self.enc_norm(x))

    def decode(self, decoder_input_ids, enc_out, input_ids, last_only=False):
        T = decoder_input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]
        causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool, device=enc_out.device))[None, None]
        position_bias = self.dec_pos_bias(T, T, enc_out.device)
        x = self.dropout(self.shared_emb(decoder_input_ids))
        for layer in self.dec_layers:
            x = layer(x, enc_out, position_bias, causal_allowed, src_allowed)
        x = self.dropout(self.dec_norm(x))
        if last_only:
            x = x[:, -1:, :]
        return x @ self.shared_emb.weight.t()                                # weight sharing

    def forward(self, input_ids, decoder_input_ids, labels=None):
        enc_out = self.encode(input_ids)
        logits = self.decode(decoder_input_ids, enc_out, input_ids)
        loss = None
        if labels is not None:
            # decoder_input_ids are already shifted (<s> + target[:-1]); -100 = padding, ignored
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}

### 7.5 Load a Phase-1 model, add the sentence classifier and the adaptation method

* **Token states:** we run the Phase-1 model without its LM output layer (for `t5` only the bidirectional encoder).
* **Sentence vector:** the mean of the token states of the sentence (`SENT_POOL = "mean"`), or the state of its `</s>` token (`"last"`, which has read the whole sentence in a left-to-right model).
* **Sentence context (`SENT_CONTEXT = "bigru"`):** two GRUs (the Phase-1 `GRUCell`, Module 2.1), one reading the sentence vectors of a window left-to-right and one right-to-left; their states are concatenated, like a bidirectional RNN (Module 2.1). This gives every sentence information about the sentences before *and after* it, also for the left-to-right Phase-1 models.
* **Head:** dropout + linear layer -> 13 labels, cross entropy with class weights `sqrt(N / count)`, capped at 2x the average (`CLASS_WEIGHTING`, `CLASS_WEIGHT_CAP`).
* **LoRA** [3]: `h = W x + (alpha/r) B A x`. **rsLoRA** [11]: the same, with scale `alpha / sqrt(r)` instead of `alpha / r`.
* **LoRMA** [12] (additive identity): `h = (I + (alpha/r) B A) W x`.
* **IA3** [13]: learned vectors (starting at 1) rescale the keys and values of every attention layer and the hidden units of every feed-forward layer: `k = l_k * (W_k x)`, `v = l_v * (W_v x)`, `FFN = W_2 (l_ff * a)`. For the RNN (no attention) the gate pre-activations `x2g` are rescaled instead.
* **KD** [2]: as in Phase 2, on the sentence label distributions: `loss = alpha * CE(gold) + (1 - alpha) * T^2 * KL(teacher || student)`.

In [ ]:
import re
from safetensors.torch import load_file

def build_base(model_type, n_layers=None):
  V = len(bpe.itos)
  if model_type == "gpt":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="learned", norm_type="layernorm", ffn_type="gelu"))
  if model_type == "modern":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_KV_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="rope", norm_type="rmsnorm", ffn_type="swiglu"))
  if model_type == "rnn":
    return RNNLM(V, RNN_HIDDEN, n_layers or RNN_LAYERS, cell_type=RNN_CELL, dropout=DROPOUT)
  if model_type == "t5":
    return T5Model(V + T5_N_SENTINELS, bpe.pad_id, T5_D_MODEL, n_layers or T5_ENC_LAYERS, T5_DEC_LAYERS,
                   T5_HEADS, T5_D_FF, dropout=DROPOUT)
  raise ValueError(model_type)

def hidden_size(model_type):
  return {"gpt": N_EMBD, "modern": N_EMBD, "rnn": RNN_HIDDEN, "t5": T5_D_MODEL}[model_type]

def load_phase1_weights(base, model_type, n_layers=None):
  path = PHASE1_MODEL_FILES[model_type]
  state = load_file(path)
  if n_layers is not None:          # student: keep only the first n_layers layers
    kept = {}
    for k, v in state.items():
      m = re.match(r"^(blocks|cells|enc_layers)\.(\d+)\.", k)
      if m and int(m.group(2)) >= n_layers:
        continue
      kept[k] = v
    state = kept
  missing, unexpected = base.load_state_dict(state, strict=False)
  print(f"loaded Phase-1 {model_type} weights from {path} | missing {len(missing)} | unexpected {len(unexpected)}")
  if missing:
    raise ValueError(f"Phase-1 weights do not match the architecture, missing: {missing[:5]}")

def lm_hidden(base, input_ids):
  # DecoderLM.forward without the LM output layer
  B, T = input_ids.shape
  pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
  x = base.tok_emb(input_ids)
  if base.cfg.pos_emb_type == "learned":
    x = x + base.pos_emb(pos)
  x = base.drop(x)
  for block in base.blocks:
    x, _ = block(x, pos)
  return base.ln_f(x)

def rnn_hidden(base, input_ids):
  # RNNLM.forward without the LM output layer
  B, T = input_ids.shape
  x = base.dropout(base.tok_emb(input_ids))
  for l in range(base.num_layers):
    cell = base.cells[l]
    h = torch.zeros(B, base.hidden_size, device=x.device, dtype=x.dtype)
    state = (h, torch.zeros_like(h)) if base.is_lstm else h
    outputs = []
    for t in range(T):
      state = cell(x[:, t, :], state)
      outputs.append(state[0] if base.is_lstm else state)
    x = base.dropout(torch.stack(outputs, dim=1))
  return x


class LoRALinear(nn.Module):
  """LoRA: h = W x + b + scale * B A x  (W, b frozen); scale = alpha / r (LoRA) or alpha / sqrt(r) (rsLoRA)"""
  def __init__(self, base_linear, r, alpha, rank_stabilized=False):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.in_features) / math.sqrt(base_linear.in_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class RSLoRALinear(LoRALinear):
  def __init__(self, base_linear, r, alpha):
    super().__init__(base_linear, r, alpha, rank_stabilized=True)


class LoRMALinear(nn.Module):
  """LoRMA (additive identity): h = (I + (alpha / r) * B A) W x + b  (W, b frozen)"""
  def __init__(self, base_linear, r, alpha):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.out_features) / math.sqrt(base_linear.out_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    y = F.linear(x, self.base.weight)                      # W x
    y = y + self.scale * ((y @ self.A.t()) @ self.B.t())   # (I + s B A) W x
    if self.base.bias is not None:
      y = y + self.base.bias
    return y


class IA3Linear(nn.Module):
  """IA3: a learned vector l (starts at 1) rescales the output ("out": keys, values)
  or the input ("in": the hidden units going into the FFN down-projection). W frozen."""
  def __init__(self, base_linear, where):
    super().__init__()
    self.base = base_linear
    self.where = where
    n = base_linear.out_features if where == "out" else base_linear.in_features
    self.l = nn.Parameter(torch.ones(n))

  def forward(self, x):
    if self.where == "out":
      return self.base(x) * self.l
    return self.base(x * self.l)


def add_adapters(module, adapter_cls, r, alpha):
  # replace every nn.Linear inside the Phase-1 model by an adapter around it
  for name, child in module.named_children():
    if isinstance(child, nn.Linear):
      setattr(module, name, adapter_cls(child, r, alpha))
    else:
      add_adapters(child, adapter_cls, r, alpha)

IA3_OUT_NAMES = ("k_proj", "v_proj", "x2g")   # keys, values (and the RNN input gates)
IA3_IN_NAMES = ("w2", "fc2", "2")             # FFN down-projections: SwiGLU w2, T5 fc2, GELU FFN net[2]

def add_ia3(module):
  for name, child in module.named_children():
    if isinstance(child, nn.Linear):
      if name in IA3_OUT_NAMES:
        setattr(module, name, IA3Linear(child, "out"))
      elif name in IA3_IN_NAMES:
        setattr(module, name, IA3Linear(child, "in"))
    else:
      add_ia3(child)


class RRModel(nn.Module):
  def __init__(self, base, model_type, d_model, n_labels, dropout, class_weights):
    super().__init__()
    self.base = base
    self.model_type = model_type
    self.dropout = nn.Dropout(dropout)
    if SENT_CONTEXT == "bigru":
      self.gru_fw = GRUCell(d_model, SENT_GRU_HIDDEN)      # reads the sentences left -> right
      self.gru_bw = GRUCell(d_model, SENT_GRU_HIDDEN)      # reads the sentences right -> left
      head_in = 2 * SENT_GRU_HIDDEN
    else:
      head_in = d_model
    self.head = nn.Linear(head_in, n_labels)
    self.register_buffer("class_weights", class_weights, persistent=False)

  def hidden(self, input_ids):
    if self.model_type == "t5":
      return self.base.encode(input_ids)                  # bidirectional encoder, padding masked
    if self.model_type == "rnn":
      return rnn_hidden(self.base, input_ids)
    return lm_hidden(self.base, input_ids)                # causal: right padding never affects real tokens

  def sentence_vectors(self, h, sent_index):
    # M[b, s, t] = 1 if token t of window b belongs to sentence s
    B, T, D = h.shape
    s_ids = torch.arange(MAX_SENTS, device=h.device).view(1, MAX_SENTS, 1)
    M = (sent_index.unsqueeze(1) == s_ids).to(h.dtype)          # (B, S, T)
    if SENT_POOL == "mean":
      counts = M.sum(dim=-1, keepdim=True).clamp(min=1)
      return (M @ h) / counts                                   # (B, S, D)
    positions = torch.arange(T, device=h.device).view(1, 1, T).to(h.dtype)
    last = (M * positions).amax(dim=-1).long()                  # position of the last token of each sentence
    return torch.gather(h, 1, last.unsqueeze(-1).expand(B, MAX_SENTS, D))

  def sentence_context(self, v, present):
    # bidirectional GRU over the sentences of each window; padding sentences do not change the state
    B, S, D = v.shape
    m = present.to(v.dtype).unsqueeze(-1)                       # (B, S, 1)
    h = v.new_zeros(B, SENT_GRU_HIDDEN)
    fw = []
    for s in range(S):
      h = m[:, s] * self.gru_fw(v[:, s], h) + (1 - m[:, s]) * h
      fw.append(h)
    h = v.new_zeros(B, SENT_GRU_HIDDEN)
    bw = [None] * S
    for s in range(S - 1, -1, -1):
      h = m[:, s] * self.gru_bw(v[:, s], h) + (1 - m[:, s]) * h
      bw[s] = h
    return torch.cat([torch.stack(fw, dim=1), torch.stack(bw, dim=1)], dim=-1)

  def forward(self, input_ids, sent_index, labels=None):
    h = self.hidden(input_ids)
    v = self.sentence_vectors(h, sent_index)
    if SENT_CONTEXT == "bigru":
      present = (sent_index.unsqueeze(1) == torch.arange(MAX_SENTS, device=h.device).view(1, MAX_SENTS, 1)).any(dim=-1)
      v = self.sentence_context(self.dropout(v), present)
    logits = self.head(self.dropout(v))                          # (B, S, n_labels)
    loss = None
    if labels is not None:
      loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1),
                             weight=self.class_weights, ignore_index=-100)
    return {"loss": loss, "logits": logits}


def set_trainable(rr, ft_method):
  for p in rr.parameters():
    p.requires_grad = (ft_method == "full")
  for name, p in rr.named_parameters():
    if not name.startswith("base."):                 # sentence GRU + head are new: always trained
      p.requires_grad = True
  if ft_method in ("lora", "rslora", "lorma"):
    for name, p in rr.base.named_parameters():
      if name.endswith(".A") or name.endswith(".B"):
        p.requires_grad = True
  if ft_method == "ia3":
    for name, p in rr.base.named_parameters():
      if name.endswith(".l"):
        p.requires_grad = True
  if ft_method == "partial":
    if rr.model_type in ("gpt", "modern"):
      layers, final_norm = rr.base.blocks, rr.base.ln_f
    elif rr.model_type == "rnn":
      layers, final_norm = rr.base.cells, None
    else:
      layers, final_norm = rr.base.enc_layers, rr.base.enc_norm
    for layer in list(layers)[-PARTIAL_TOP_LAYERS:]:
      for p in layer.parameters():
        p.requires_grad = True
    if final_norm is not None:
      for p in final_norm.parameters():
        p.requires_grad = True


def label_weights():
  w = torch.ones(len(LABELS))
  if CLASS_WEIGHTING == "sqrt_inverse":
    c = Counter(l for x in train_docs for l in x["labels"])
    N = sum(c.values())
    for l, i in LABEL2ID.items():
      w[i] = math.sqrt(N / max(c.get(l, 0), 1))
    w = w / w.mean()
    w = w.clamp(max=CLASS_WEIGHT_CAP)
  return w


def build_rr_model(model_type, ft_method, n_layers=None, load_pretrained=True):
  base = build_base(model_type, n_layers)
  if load_pretrained:
    load_phase1_weights(base, model_type, n_layers)
  if model_type == "t5":
    base.dec_layers = nn.ModuleList()       # RR uses only the T5 encoder
  if ft_method == "lora":
    add_adapters(base, LoRALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method == "rslora":
    add_adapters(base, RSLoRALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method == "lorma":
    add_adapters(base, LoRMALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method == "ia3":
    add_ia3(base)
  rr = RRModel(base, model_type, hidden_size(model_type), len(LABELS), DROPOUT, label_weights())
  set_trainable(rr, ft_method)
  return rr


class KDModel(nn.Module):
  def __init__(self, student, teacher, temperature, alpha):
    super().__init__()
    self.student = student
    self.teacher = teacher
    self.T = temperature
    self.alpha = alpha
    for p in self.teacher.parameters():
      p.requires_grad = False

  def train(self, mode=True):
    super().train(mode)
    self.teacher.eval()                     # the teacher never uses dropout
    return self

  def forward(self, input_ids, sent_index, labels=None):
    out = self.student(input_ids, sent_index, labels)
    if labels is None:
      return out
    with torch.no_grad():
      t_logits = self.teacher(input_ids, sent_index)["logits"]
    mask = labels != -100
    s = out["logits"][mask].float()
    t = t_logits[mask].float()
    kd = F.kl_div(F.log_softmax(s / self.T, dim=-1), F.softmax(t / self.T, dim=-1),
                  reduction="batchmean") * self.T * self.T
    return {"loss": self.alpha * out["loss"] + (1 - self.alpha) * kd, "logits": out["logits"]}


print("label weights:", {l: round(float(w), 2) for l, w in zip(LABELS, label_weights())})
if USE_KD:
  student = build_rr_model(MODEL_TYPE, FT_METHOD, n_layers=STUDENT_LAYERS)
  teacher = build_rr_model(TEACHER_MODEL_TYPE, TEACHER_FT_METHOD, load_pretrained=False)
  teacher.load_state_dict(torch.load(TEACHER_RR_PATH, map_location="cpu"))
  print("teacher loaded from", TEACHER_RR_PATH)
  model = KDModel(student, teacher, KD_TEMPERATURE, KD_ALPHA)
  rr_model = student
  n_teacher = sum(p.numel() for p in teacher.parameters())
else:
  rr_model = build_rr_model(MODEL_TYPE, FT_METHOD, load_pretrained=USE_PHASE1_WEIGHTS)
  model = rr_model
  n_teacher = 0

n_total = sum(p.numel() for p in rr_model.parameters())
n_trainable = sum(p.numel() for p in rr_model.parameters() if p.requires_grad)
print(f"RR model: {n_total / 1e6:.2f}M parameters | trainable {n_trainable / 1e6:.3f}M ({100 * n_trainable / n_total:.2f}%)")
if USE_KD:
  print(f"teacher: {n_teacher / 1e6:.2f}M parameters -> student is {100 * n_total / n_teacher:.0f}% of the teacher")

## 8. Training (Trainer, like Phase 1 and 2)

The checkpoint is chosen by the **macro-F1 of the dev windows** (the task metric) rather than by the dev loss: in Phase 2 the dev loss started rising after epoch 2 while the predictions still changed.

In [ ]:
from transformers import TrainingArguments, Trainer

def macro_scores(gold, pred):
  # macro P / R / F1 over the labels that appear in the gold or the predictions
  labels = sorted(set(gold) | set(pred))
  per = {}
  for l in labels:
    tp = sum(1 for g, p in zip(gold, pred) if g == l and p == l)
    fp = sum(1 for g, p in zip(gold, pred) if g != l and p == l)
    fn = sum(1 for g, p in zip(gold, pred) if g == l and p != l)
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    per[l] = (P, R, F1, tp + fn)
  n = max(len(per), 1)
  return (sum(v[0] for v in per.values()) / n, sum(v[1] for v in per.values()) / n,
          sum(v[2] for v in per.values()) / n, per)

def preprocess_logits_for_metrics(logits, labels):
  if isinstance(logits, (tuple, list)):
    logits = logits[0]
  return logits.argmax(dim=-1)

def compute_metrics(eval_pred):
  pred = eval_pred.predictions.reshape(-1)
  gold = eval_pred.label_ids.reshape(-1)
  keep = gold != -100
  return {"macro_f1": macro_scores(gold[keep].tolist(), pred[keep].tolist())[2]}

training_args = TrainingArguments(
  output_dir=os.path.join(EXP_DIR, "checkpoints"),
  num_train_epochs=EPOCHS,
  per_device_train_batch_size=BATCH_SIZE,
  per_device_eval_batch_size=BATCH_SIZE,
  learning_rate=LEARNING_RATE,
  weight_decay=0.01,
  warmup_steps=5 if SMOKE_TEST else 50,
  lr_scheduler_type="cosine",
  max_grad_norm=1.0,
  fp16=torch.cuda.is_available() and MODEL_TYPE != "t5",   # T5 trains in full precision (fp16 broke it in Phase 2)
  eval_strategy="epoch",
  save_strategy="epoch",
  save_total_limit=2,
  load_best_model_at_end=True,          # checkpoint selection on dev macro-F1
  metric_for_best_model="macro_f1",
  greater_is_better=True,
  logging_steps=20,
  seed=SEED,
  report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,
                  eval_dataset=val_dataset, data_collator=data_collator,
                  compute_metrics=compute_metrics, preprocess_logits_for_metrics=preprocess_logits_for_metrics)

if torch.cuda.is_available():
  torch.cuda.reset_peak_memory_stats()
start_time = time.time()
if LOAD_RR_PATH:
  rr_model.load_state_dict(torch.load(LOAD_RR_PATH, map_location="cpu"))
  print("loaded trained RR model from", LOAD_RR_PATH, "- skipping training")
else:
  trainer.train()
training_time = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"Training time: {training_time / 60:.1f} min | peak GPU memory: {peak_memory_gb:.2f} GB")
for h in trainer.state.log_history:
  if "eval_loss" in h:
    print(f"epoch {h['epoch']:.0f}: eval_loss {h['eval_loss']:.4f} | eval macro-F1 {h.get('eval_macro_f1', float('nan')):.4f}")
  elif "loss" in h:
    print(f"   step {h['step']}: train loss {h['loss']:.4f}")

## 9. Dev evaluation: macro-F1 per domain and error analysis

Prediction runs on whole dev documents, exactly like the test set: every sentence gets exactly one label. We report macro-F1 for CL, IT and both together, the per-label scores in each domain, the most confused label pairs, and a majority-label baseline for reference.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rr_model = rr_model.to(device)
rr_model.eval()
PRED_BATCH = 16
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

@torch.no_grad()
def predict_doc(x):
  st = doc_tok[(x["domain"], x["id"])]
  n = len(st)
  best_score = [-1] * n
  best_label = [0] * n
  wins = make_windows(st)
  for b in range(0, len(wins), PRED_BATCH):
    chunk = wins[b:b + PRED_BATCH]
    ids, sidx = [], []
    for s, e in chunk:
      a, c, _ = window_example(st, None, s, e)
      ids.append(a)
      sidx.append(c)
    ids = torch.tensor(ids, dtype=torch.long, device=device)
    sidx = torch.tensor(sidx, dtype=torch.long, device=device)
    if device.type == "cuda" and MODEL_TYPE != "t5":
      with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = rr_model(ids, sidx)["logits"]
    else:
      logits = rr_model(ids, sidx)["logits"]
    pred = logits.argmax(dim=-1).tolist()
    for j, (s, e) in enumerate(chunk):
      L = e - s
      for k in range(L):
        if CAUSAL and SENT_CONTEXT == "none":
          score = k                       # left-to-right model: the window with the most sentences on the left
        else:
          score = min(k, L - 1 - k)       # the window where the sentence is most central
        if score > best_score[s + k]:
          best_score[s + k] = score
          best_label[s + k] = pred[j][k]
  return [ID2LABEL[i] for i in best_label]

t0 = time.time()
dev_pred = {(x["domain"], x["id"]): predict_doc(x) for x in dev_docs}
dev_seconds = time.time() - t0

def domain_lists(domain):
  g, p = [], []
  for x in dev_docs:
    if domain is None or x["domain"] == domain:
      g += x["labels"]
      p += dev_pred[(x["domain"], x["id"])]
  return g, p

dev_scores = {}
for d in DOMAINS + [None]:
  g, p = domain_lists(d)
  dev_scores[d or "ALL"] = macro_scores(g, p)
  name = d or "CL+IT"
  P, R, F1, _ = dev_scores[d or "ALL"]
  maj = Counter(g).most_common(1)[0][0]
  base_f1 = macro_scores(g, [maj] * len(g))[2]
  print(f"DEV {name:6s}: macro P {P:.4f} | R {R:.4f} | F1 {F1:.4f}   ({len(g)} sentences; majority-label baseline F1 {base_f1:.4f})")
mean_domain_f1 = sum(dev_scores[d][2] for d in DOMAINS) / len(DOMAINS)
print(f"mean of the CL and IT macro-F1: {mean_domain_f1:.4f}   ({dev_seconds:.1f}s for {len(dev_docs)} dev documents)")

print(f"\n{'label':24s} " + " ".join(f"{d + ' F1':>7s} {d + ' #':>5s}" for d in DOMAINS) + f" {'ALL P':>6s} {'ALL R':>6s} {'ALL F1':>6s}")
per_all = dev_scores["ALL"][3]
for l in sorted(per_all, key=lambda l: per_all[l][2]):
  row = f"{l:24s} "
  for d in DOMAINS:
    v = dev_scores[d][3].get(l, (0, 0, 0, 0))
    row += f"{v[2]:7.3f} {v[3]:5d} "
  row += f"{per_all[l][0]:6.3f} {per_all[l][1]:6.3f} {per_all[l][2]:6.3f}"
  print(row)

# ---- error analysis: which roles are confused with which ----
confusions = Counter()
for d in DOMAINS:
  g, p = domain_lists(d)
  for a, b in zip(g, p):
    if a != b:
      confusions[(d, a, b)] += 1
print("\nmost confused (domain, gold -> predicted):")
for (d, a, b), c in confusions.most_common(15):
  print(f"  {d}  {a:22s} -> {b:22s} {c}")

CLOSE_PAIRS = [("ArgumentPetitioner", "ArgumentRespondent"), ("PrecedentReliedUpon", "PrecedentNotReliedUpon"),
               ("PrecedentReliedUpon", "PrecedentOverruled"), ("RatioOfTheDecision", "RulingByPresentCourt"),
               ("Fact", "RulingByLowerCourt"), ("Fact", "Issue"), ("RatioOfTheDecision", "PrecedentReliedUpon")]
print("\nclosely related roles (mistakes in both directions, CL / IT):")
for a, b in CLOSE_PAIRS:
  if a in LABEL2ID and b in LABEL2ID:
    cl = confusions[("CL", a, b)] + confusions[("CL", b, a)]
    it = confusions[("IT", a, b)] + confusions[("IT", b, a)]
    print(f"  {a:22s} <-> {b:22s}  CL {cl:4d} | IT {it:4d}")

# ---- does position in the document matter? accuracy by position (first / middle / last third) ----
for d in DOMAINS:
  parts = {"first third": [0, 0], "middle third": [0, 0], "last third": [0, 0]}
  for x in dev_docs:
    if x["domain"] != d:
      continue
    p = dev_pred[(x["domain"], x["id"])]
    n = len(p)
    for i, (a, b) in enumerate(zip(x["labels"], p)):
      key = "first third" if i < n / 3 else ("middle third" if i < 2 * n / 3 else "last third")
      parts[key][0] += int(a == b)
      parts[key][1] += 1
  print(f"{d} accuracy by position:", {k: round(v[0] / max(v[1], 1), 3) for k, v in parts.items()})

## 10. Save the RR model (needed when this experiment is the KD teacher)

In [ ]:
torch.save(rr_model.state_dict(), os.path.join(EXP_DIR, "rr_model.pt"))
print("saved", os.path.join(EXP_DIR, "rr_model.pt"))

## 11. Test predictions -> prediction.json -> submission zip

One file maps every test document id (CL and IT together) to its ordered list of labels. The CL and IT predictions are also saved separately in the experiment folder.

In [ ]:
t0 = time.time()
predictions = {}
per_domain = {d: {} for d in DOMAINS}
for x in test_docs:
  labs = predict_doc(x)
  assert len(labs) == len(x["sents"])               # exactly one label per sentence, same order
  assert all(l in LABEL2ID for l in labs)
  if x["id"] in predictions:
    print("WARNING: document id", x["id"], "appears in more than one test file")
  predictions[x["id"]] = labs
  per_domain[x["domain"]][x["id"]] = labs
test_seconds = time.time() - t0

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
  json.dump(predictions, f, ensure_ascii=False)
for d in DOMAINS:
  with open(os.path.join(EXP_DIR, f"prediction_{d}.json"), "w", encoding="utf-8") as f:
    json.dump(per_domain[d], f, ensure_ascii=False)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
  z.write(PREDICTION_PATH, arcname="prediction.json")
print(f"{sum(len(v) for v in predictions.values())} sentences labelled in {len(predictions)} test documents in {test_seconds:.0f}s")
print("test label distribution:", dict(Counter(l for v in predictions.values() for l in v).most_common()))
print("submission:", SUBMISSION_ZIP)

## 12. Log this experiment (one line per experiment in results.jsonl)

In [ ]:
result = {
  "experiment": EXPERIMENT_NAME, "model": MODEL_TYPE, "method": FT_METHOD, "kd": USE_KD,
  "student_layers": STUDENT_LAYERS if USE_KD else None,
  "teacher": f"{TEACHER_MODEL_TYPE}/{TEACHER_FT_METHOD}" if USE_KD else None,
  "phase1_weights": USE_PHASE1_WEIGHTS, "train_domains": TRAIN_DOMAINS,
  "context": CONTEXT_MODE, "sent_context": SENT_CONTEXT, "pool": SENT_POOL, "class_weighting": CLASS_WEIGHTING,
  "window": WINDOW_TOKENS, "max_sents": MAX_SENTS,
  "params_M": round(n_total / 1e6, 2), "trainable_M": round(n_trainable / 1e6, 3),
  "trainable_pct": round(100 * n_trainable / n_total, 2),
  "train_min": round(training_time / 60, 1), "peak_mem_GB": round(peak_memory_gb, 2),
  "test_pred_s": round(test_seconds, 1),
  "dev_F1_CL": round(dev_scores["CL"][2], 4), "dev_F1_IT": round(dev_scores["IT"][2], 4),
  "dev_F1_all": round(dev_scores["ALL"][2], 4), "dev_P_all": round(dev_scores["ALL"][0], 4),
  "dev_R_all": round(dev_scores["ALL"][1], 4), "dev_F1_mean_domains": round(mean_domain_f1, 4),
  "per_label_F1": {l: round(v[2], 3) for l, v in dev_scores["ALL"][3].items()},
  "lr": LEARNING_RATE, "epochs": EPOCHS,
  "rank": ADAPTER_RANK if FT_METHOD in ("lora", "rslora", "lorma") else None,
}
with open(RESULTS_PATH, "a") as f:
  f.write(json.dumps(result) + "\n")
print(json.dumps(result, indent=1))